In [0]:
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("storage_acount", ".", "Adls")
storage_acount = dbutils.widgets.get("storage_acount")
dbutils.widgets.text("container", ".", "container")
container = dbutils.widgets.get("container")
dbutils.widgets.text("catalog", ".", "catalog")
catalog = dbutils.widgets.get("catalog")
dbutils.widgets.text("container_s", ".", "container_s")
silver = dbutils.widgets.get("container_s")
dbutils.widgets.text("container_g", ".", "container_g")
gold = dbutils.widgets.get("container_g")

In [0]:
customers_sv = spark.table(f"{catalog}.{silver}.customers")
loans_sv     = spark.table(f"{catalog}.{silver}.loans")
payments_sv  = spark.table(f"{catalog}.{silver}.payments")
customer_loan_summary_sv = spark.table(f"{catalog}.{silver}.customer_loan_summary")

In [0]:
customers_g = (
    customers_sv
    .withColumn("full_name", F.concat_ws(" ", F.col("first_name"), F.col("last_name")))
    .withColumn("age", F.floor(F.datediff(F.current_date(), F.col("date_of_birth"))/365.25))
    .withColumn("missing_email", F.when(F.col("email").isNull(), 1).otherwise(0))
    .withColumn("missing_phone", F.when(F.col("phone_number").isNull(), 1).otherwise(0))
)

In [0]:
loans_g = (
    loans_sv
    .withColumn("term_months", F.months_between(F.col("end_date"), F.col("start_date")))
    .withColumn("pct_paid", F.col("total_paid") / F.col("loan_amount"))
    .withColumn("balance_due", F.col("loan_amount") - F.col("total_paid"))
)

In [0]:
payments_g = (
    payments_sv
    .withColumn("is_late", F.when(F.col("payment_status") != "PAID", 1).otherwise(0))
)

In [0]:
customer_loan_summary_g = (
    customer_loan_summary_sv
    .groupBy("customer_id", "first_name", "last_name", "email", "phone_number")
    .agg(
        F.count("loan_id").alias("num_loans"),
        F.sum("loan_amount").alias("total_loan_amount"),
        F.sum("total_paid").alias("total_paid"),
        F.sum("balance_due").alias("total_balance_due"),
        F.max("last_payment_date").alias("last_payment_date")
    )
    .withColumn("pct_total_paid", F.col("total_paid") / F.col("total_loan_amount"))
)

In [0]:
customers_g.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold}.customers")
loans_g.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold}.loans")
payments_g.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold}.payments")
customer_loan_summary_g.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{gold}.customer_loan_summary")